# The Gaia-like kernel: the digitized LSF of Rowell et al. (2021), and its derivatives

**In the paper:** Sect. 6.2 (the digitized LSF, linear in ν_eff, its footnote; FWHM 110 → 100 mas; 80 % within one FWHM); Fig. 5b (the K star's kernel and its colour derivative).

Section 6 and Figures 5 and 6 replace the toy Gaussian by a Gaia-like
along-scan line spread function (LSF) whose shape depends on the star's
colour. This notebook builds that kernel from the digitized data, step by
step, shows every choice made on the way, and computes the two derivatives
the paper uses: with respect to the source position (the fits, the Fisher
matrix) and with respect to the colour (Fig. 5).

**The source and its assumptions.**

- Rowell et al. (2021, A&A 649, A11), Fig. 11: the calibrated along-scan LSF of
  the AF2–9 CCDs, **window class 1**, at **revolution 3343**, for five effective
  wavenumbers ν_eff = 1.24, 1.36, 1.48, 1.60, 1.72 µm⁻¹, for each telescope.
  The paper uses this one calibration for every magnitude and epoch.
- The figure is a raster in the published PDF. It was digitized by the colour
  of each curve, column by column (by the script `digitize_fig11.py` beside the data),
  into `data/rowell2021_lsf/lsf_af29_wc1.csv`; that folder's README
  holds the provenance. The axes were calibrated on the major ticks and the
  caption's vertical offsets (0.02 per curve) removed. Drawn back onto the
  original image with the offsets restored, the digitized points lie on
  Rowell's curves over their whole length.
- **One telescope, FOV1.** The file holds both; FOV2 has a diffraction
  shoulder at −2.5 pixels and is not used.
- **Linear in ν_eff** between the tabulated colours (Sect. 6.2); no
  extrapolation outside 1.24–1.72 µm⁻¹.
- One along-scan pixel is the toy pitch, 59 mas (Gaia's is 58.9), so the
  truth and the model share one mapping.

| step | task | calls | paper |
|---|---|---|---|
| 1 | the raw digitized profiles | — (reads `data/rowell2021_lsf/`) | Sect. 6.2 |
| 2 | smoothing | — | — |
| 3 | the centre | — | — |
| 4 | the wings beyond the plotted ±9 pixels, and the total flux | — (the notebook's class `RowellLSF`) | Sect. 6.2 |
| 5 | the interpolation in colour, and what the profiles say | — | Sect. 6.2 and its footnote |
| 6 | the position derivative | — | — |
| 7 | the colour derivative | — | Fig. 5 |
| 8 | the kernel on the toy window | `ins.Instrument` | Sect. 6.2 |
| 9 | the figure | — | — |

The one scanlib name the notebook uses carries its module (`ins.Instrument`); the first code cell lists the
module.

**What it reproduces.** Sect. 6.2: the FWHM falls from about 110 to about
100 mas from ν_eff = 1.24 to 1.72 µm⁻¹; 80 % of the flux lies within one FWHM
of the centre, against 98 % for a Gaussian; the footnote's accuracy, the
interpolation good to about 1 % of the peak, and neighbouring profiles near
1.42 µm⁻¹ that differ by three to four times that. Fig. 5: the K star's
kernel (ν_eff = 1.42 µm⁻¹) and its colour derivative, the secant of the
profiles at 1.36 and 1.48 µm⁻¹. Run time: about 2 s.

In [ ]:
# the imports: numpy, matplotlib, scipy's splines, and the scanlib module; every scanlib call below carries its
# module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline, UnivariateSpline

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in sim_margins_lambda_w)
FWHM_MAS, PITCH_MAS, N_SAMPLES = 100.0, 59.0, 18
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)

In [ ]:
# the switch SAVE_PDF, the data file, and the constants of the kernel
SAVE_PDF = False                           # True: write pdf/kernel_rowell_lsf.pdf
CSV = Path("..") / ".." / ".." / "data" / "rowell2021_lsf" / "lsf_af29_wc1.csv"

FOV = "FOV1"                               # the telescope used
NU = (1.24, 1.36, 1.48, 1.60, 1.72)        # [1/micron] the tabulated effective wavenumbers
NU_K = 1.42                                # [1/micron] the K star of Sect. 6 and Figs. 5, 6
PIXEL_MAS = PITCH_MAS                      # one along-scan pixel [mas], the toy pitch
# COLOURS: a table from a tabulated colour to the shade of gray of its curve in the figure ("0.0" is black,
# "1.0" white), the paper's style: COLOURS[1.24], the reddest colour, is black. zip pairs the five colours of NU
# with the five shades, in order
COLOURS = dict(zip(NU, ("0.0", "0.25", "0.45", "0.6", "0.75")))

## 1. The raw digitized profiles

Each curve was read off the image column by column: 468 columns over ±9
pixels. One image row is 0.5/293 = 0.0017 per pixel in the LSF (0.4 % of the
peak), one image column 0.038 pixels (2.3 mas). The values are as digitized:
not smoothed, not renormalized, not recentred.

In [ ]:
# read the CSV's rows (fov, nu_eff, x, L); digitized(fov) gives one telescope's curves: per tabulated colour,
# x [pixels] and L [per pixel].
# rows: one row per digitized point, a list of four texts (fov, nu_eff, x [pixels], L [per pixel]); the comment
# lines (they start with '#') and the empty lines are skipped, and [1:] omits the header row
rows = [l.split(",") for l in CSV.read_text().splitlines() if l and not l.startswith("#")][1:]
ROW_LSF = 0.5 / 293.0                      # one image row in the LSF [per pixel]


def digitized(fov):
    """``{nu: (x [pixels], L [per pixel])}`` of one telescope, as digitized."""
    out = {}
    for nu in NU:
        # r[0] the telescope, r[1] nu_eff, r[2] the offset x, r[3] the LSF: keep this telescope and this colour
        sel = [r for r in rows if r[0] == fov and abs(float(r[1]) - nu) < 1e-9]
        out[nu] = (np.array([float(r[2]) for r in sel]), np.array([float(r[3]) for r in sel]))
    return out

In [ ]:
# the curves of FOV1 printed (profile[0] and profile[-1] are L at the first and the last column), and the
# digitization checked: 468 columns, in increasing x (RAW[nu][0] is x), over the plotted +-9 pixels.
# RAW: a table from a tabulated colour to the pair (x, L) of its digitized curve; RAW[1.24][0] is the offsets x
# [pixels] of the reddest colour's columns, RAW[1.24][1] its LSF values L [per pixel]
RAW = digitized(FOV)
for nu in NU:
    offsets, profile = RAW[nu]             # the offsets of the columns [pixels], and the LSF there [per pixel]
    column_step = np.diff(offsets).mean()        # the mean distance between neighbouring columns [pixels]
    print(f"nu_eff {nu:.2f}: {offsets.size} columns, x {offsets.min():+.3f} to {offsets.max():+.3f} pixels, "
          f"step {column_step:.4f}; peak {profile.max():.4f}; at -9 and +9 pixels {profile[0]:.4f}, {profile[-1]:.4f}")
# check 1: every curve has all its 468 columns
assert all(RAW[nu][0].size == 468 for nu in NU)                        # no gap in any curve
# check 2: the offsets increase from one column to the next (np.diff: each offset minus the one before it)
assert all(np.all(np.diff(RAW[nu][0]) > 0) for nu in NU)
# check 3: the largest |x| of every curve is 9 pixels to within one column
assert all(abs(np.abs(RAW[nu][0]).max() - 9.0) < 0.04 for nu in NU)   # the plotted +-9 pixels

## 2. Smoothing

The digitized curve is a staircase of image rows. A cubic smoothing spline
per curve removes it: its residual is set to the quantization of one image
row, a uniform error of standard deviation ROW_LSF/√12, so the spline takes
out the staircase and nothing more. The smoothed profiles are tabulated on a
fine grid over ±9 pixels.

In [ ]:
# smooth each curve with a cubic smoothing spline whose residual is one image row's quantization, and
# tabulate it on the fine grid X_FINE; then check that the residual is that quantization
X_MAX = 9.0                                # the digitized range [pixels]
SMOOTH_RMS = ROW_LSF / np.sqrt(12.0)       # the quantization of one image row
X_FINE = np.linspace(-X_MAX, X_MAX, 3601)  # the grid of the smoothed profiles [pixels]
# step 1, TAB: a table from a tabulated colour to its smoothed profile on X_FINE [per pixel]. A smoothing spline
# per colour (x, l: its digitized offsets and LSF values); s = (number of points) x SMOOTH_RMS^2 lets it leave
# residuals of the size of one image row's rounding
TAB = {nu: UnivariateSpline(x, l, s=x.size * SMOOTH_RMS ** 2)(X_FINE) for nu, (x, l) in RAW.items()}

# step 2, RES: a table from a tabulated colour to the residual of its smoothing [per pixel], the standard
# deviation of the digitized values minus the smoothed profile, read at the digitized columns
RES = {}
for nu in NU:
    offsets, profile = RAW[nu]             # the digitized offsets [pixels] and LSF values [per pixel]
    smoothed_at_columns = np.interp(offsets, X_FINE, TAB[nu])
    RES[nu] = float(np.std(profile - smoothed_at_columns))
for nu in NU:
    print(f"nu_eff {nu:.2f}: residual of the smoothing {RES[nu]:.5f} per pixel (quantization {SMOOTH_RMS:.5f})")
# step 3, the check: every residual is the quantization to within 10 %
assert all(abs(RES[nu] / SMOOTH_RMS - 1.0) < 0.1 for nu in NU)

## 3. The centre

The profiles are kept as digitized, not recentred. Their peaks all lie at
the same column, −0.019 pixels (about 1 mas). The profile is not symmetric:
the side at positive x carries more light (the shoulder at +2 to +3 pixels),
so the centroid within ±9 pixels lies at +0.12 pixels (7 mas), while the
centroid of the core (above half maximum) lies at −0.02 to −0.03 pixels. Each
moves with colour by about 0.01 pixels (0.6 mas) at most. The
fitted source position is the zero of the kernel, wherever the light's
centroid is; a common offset does not change a separation, and the colour
moves it by less than a milliarcsecond.

In [ ]:
# where the profiles peak, and where their light's centroid lies: over +-9 pixels and in the core.
# RAW[nu] is the pair (x, L): [0] is x and [1] is L, so PEAK_X is the x of the largest L
# step 1, PEAK_X: a table from a tabulated colour to the offset [pixels] of the digitized column with the largest L
PEAK_X = {}
for nu in NU:
    offsets, profile = RAW[nu]
    PEAK_X[nu] = float(offsets[np.argmax(profile)])        # np.argmax: the index of the largest value


def centroid(nu, mask):
    """The centroid of the light of one smoothed profile over a part of the fine grid.

    Inputs
        nu     a tabulated colour [1/micron]
        mask   one True or False per point of X_FINE: the points to use
    Returns the centroid [pixels]: the integral of x L(x) over the integral of L(x), both by the trapezoid rule
    over the selected points."""
    x_selected = X_FINE[mask]
    profile_selected = TAB[nu][mask]
    first_moment = np.trapezoid(x_selected * profile_selected, x_selected)
    flux = np.trapezoid(profile_selected, x_selected)
    return float(first_moment / flux)


# step 2, CENTROID and CORE: tables from a tabulated colour to the centroid [pixels] over the whole digitized
# range, and over the core (the points at or above half the profile's maximum)
CENTROID = {}
CORE = {}
for nu in NU:
    whole_range = np.ones_like(X_FINE, bool)                # every point of X_FINE: over +-9 pixels
    above_half_maximum = TAB[nu] >= 0.5 * TAB[nu].max()     # above half maximum
    CENTROID[nu] = centroid(nu, whole_range)
    CORE[nu] = centroid(nu, above_half_maximum)
for nu in NU:
    print(f"nu_eff {nu:.2f}: peak at {PEAK_X[nu]:+.3f} pixels; centroid of the core {CORE[nu]:+.4f}, "
          f"over +-9 pixels {CENTROID[nu]:+.4f}")
# step 3, spread: a table from the kind of centroid to its largest minus its smallest value over the five
# colours [pixels]
spread = {}
spread["core"] = max(CORE.values()) - min(CORE.values())
spread["+-9 pixels"] = max(CENTROID.values()) - min(CENTROID.values())
spread_texts = []
for kind, value in spread.items():
    spread_texts.append(f"{kind} {value:.4f} pixels ({value * PIXEL_MAS:.2f} mas)")
print("spread over the colours: " + ", ".join(spread_texts))
# step 4, the checks. The peaks: set(...) keeps the distinct values, so one value means the same column for every
# colour, and it lies within 0.02 pixels of zero. The centroids: within the ranges that the text quotes. The
# spread: below 0.012 pixels
assert len(set(PEAK_X.values())) == 1 and abs(PEAK_X[NU[0]]) < 0.02
assert all(-0.035 < c < -0.02 for c in CORE.values()) and all(0.11 < c < 0.13 for c in CENTROID.values())
assert max(spread.values()) < 0.012

## 4. The wings and the total flux

Rowell's profiles carry unit flux, the part outside the window included. The
digitized ±9 pixels hold 94–95 % of it. Beyond them each side continues as
A (9/|x|)², the Lorentzian wings of Rowell's basis, with A fitted per side
over 7–9 pixels (least squares, on the smoothed profile). Over that range the
profile blends from the spline into the tail by a cubic smoothstep, so the
kernel and its derivative are continuous.

**What fixes the wing level.** At ±9 pixels the profiles are 0.002–0.004 per
pixel, only one to three image rows above the zero line, so a zero line one
row off would change the tail by half. Two facts hold it: the zero line is
the axis of the image (the grid rows step by 58–59 rows per 0.1 down to it),
and the tail the wing level implies, 2 × 9 A, makes up the flux missing
inside ±9 pixels: the total is one to within 1 %, which nothing was fitted to
give. The shape beyond 9 pixels is the assumption: the figure does not show it.

In [ ]:
# the kernel at any colour (RowellLSF): the smoothed profiles interpolated linearly in colour, continued by
# A (9/|x|)^2 wings beyond +-9 pixels and blended into them over 7-9 pixels
X_JOIN = 7.0                               # the core blends into the tail over X_JOIN <= |x| <= X_MAX
TAIL_POWER = 2.0                           # the wings beyond X_MAX, L ~ |x|^-2


class RowellLSF:
    """The FOV1 LSF at one effective wavenumber: L(x) [per pixel] and L'(x) [per pixel^2] for x in pixels.

    Linear in nu between the tabulated colours: the core, the wing amplitudes and the blend are all linear
    in the tabulated profiles.

    The kernel has three parts: the core (a spline through the smoothed profile, within +-9 pixels), the tail
    (a wing A (9/|x|)^2 on each side), and a blend weight that goes from 0 at |x| = 7 pixels to 1 at 9 pixels:
    L(x) = (1 - weight) core + weight tail."""

    def __init__(self, nu_eff):
        if not NU[0] <= nu_eff <= NU[-1]:
            raise ValueError(f"nu_eff {nu_eff} outside the tabulated {NU[0]}-{NU[-1]} per micron")
        # step 1, the two tabulated colours that bracket nu_eff. index_below: the index of the tabulated colour at
        # or below nu_eff (the last interval for nu_eff = 1.72); weight_above: the weight of the colour above it
        index_below = min(int(np.searchsorted(NU, nu_eff, side="right")) - 1, len(NU) - 2)
        weight_above = (nu_eff - NU[index_below]) / (NU[index_below + 1] - NU[index_below])
        self.nu_eff = float(nu_eff)
        # step 2, the core: the two tabulated profiles mixed linearly, and a cubic spline through the mix
        profile = (1.0 - weight_above) * TAB[NU[index_below]] + weight_above * TAB[NU[index_below + 1]]
        self.core = CubicSpline(X_FINE, profile)
        # step 3, the wings: the amplitude A of A (9/|x|)^2 by least squares over X_JOIN <= |x| <= X_MAX, per
        # side. The two masks: the left side (x <= -7 pixels) and the right side (x >= 7 pixels) of the fine grid
        amplitudes = []
        for side in (X_FINE <= -X_JOIN, X_FINE >= X_JOIN):
            wing_shape = (X_MAX / np.abs(X_FINE[side])) ** TAIL_POWER
            amplitudes.append(float(np.sum(profile[side] * wing_shape) / np.sum(wing_shape ** 2)))
        self.edge = tuple(amplitudes)      # (the left wing's amplitude, the right wing's)

    def _tail(self, x, order=0):
        """The wing at x [pixels] (order 0) or its derivative in x (order 1)."""
        # edge[0]: the left wing's amplitude, edge[1]: the right wing's
        amplitude = np.where(x < 0, self.edge[0], self.edge[1])
        distance = np.abs(x)
        if order == 0:
            return amplitude * (X_MAX / distance) ** TAIL_POWER
        return -TAIL_POWER * np.sign(x) * amplitude * X_MAX ** TAIL_POWER / distance ** (TAIL_POWER + 1)

    def _weight(self, x, order=0):
        """The blend weight at x [pixels] (order 0) or its derivative in x (order 1): 0 for |x| <= 7, 1 for
        |x| >= 9, the smooth step 3 t^2 - 2 t^3 between, with t the fraction of the way from 7 to 9."""
        fraction = np.clip((np.abs(x) - X_JOIN) / (X_MAX - X_JOIN), 0.0, 1.0)
        if order == 0:
            return fraction * fraction * (3.0 - 2.0 * fraction)
        return np.sign(x) * 6.0 * fraction * (1.0 - fraction) / (X_MAX - X_JOIN)

    def __call__(self, x):
        """L(x) [per pixel] at x [pixels]."""
        x = np.asarray(x, dtype=float)
        # the core is evaluated within +-9 pixels only, the tail outside +-7 pixels only; where a part has the
        # weight zero its argument is moved to a harmless value (the spline is not extrapolated, 1/|x| stays finite)
        x_core = np.clip(x, -X_MAX, X_MAX)
        x_tail = np.where(np.abs(x) < X_JOIN, X_JOIN, x)
        weight = self._weight(x)
        return (1.0 - weight) * self.core(x_core) + weight * self._tail(x_tail)

    def derivative(self, x):
        """L'(x) [per pixel^2] at x [pixels]: the derivative of (1 - weight) core + weight tail, by the product rule."""
        x = np.asarray(x, dtype=float)
        x_core = np.clip(x, -X_MAX, X_MAX)
        x_tail = np.where(np.abs(x) < X_JOIN, X_JOIN, x)
        weight = self._weight(x)
        weight_slope = self._weight(x, 1)
        # self.core(x_core, 1): the spline's first derivative
        return ((1.0 - weight) * self.core(x_core, 1) + weight * self._tail(x_tail, 1)
                + weight_slope * (self._tail(x_tail) - self.core(x_core)))

    def flux(self):
        """The profile over +-9 pixels plus the two tails integrated to infinity."""
        return float(np.trapezoid(self(X_FINE), X_FINE) + sum(self.edge) * X_MAX / (TAIL_POWER - 1.0))

    def fwhm_mas(self):
        """The full width at half maximum [mas], from the core."""
        profile = self.core(X_FINE)
        half_maximum = 0.5 * profile.max()
        # above_half: the indices of the fine grid at or above half maximum; [0] and [-1] are the first and the
        # last, and the two crossings are interpolated linearly between each and its neighbour outside
        above_half = np.where(profile >= half_maximum)[0]
        first = above_half[0]
        last = above_half[-1]
        left_crossing = np.interp(half_maximum, [profile[first - 1], profile[first]], [X_FINE[first - 1], X_FINE[first]])
        right_crossing = np.interp(half_maximum, [profile[last + 1], profile[last]], [X_FINE[last + 1], X_FINE[last]])
        return float((right_crossing - left_crossing) * PIXEL_MAS)

In [ ]:
# the kernel at each tabulated colour; the flux inside +-9 pixels plus the two tails must be one to within 1 %.
# RAW[nu][1][0] and RAW[nu][1][-1] are L at the first and the last digitized column, -9 and +9 pixels
# step 1, LSF: a table from a tabulated colour to its kernel (a RowellLSF); INSIDE: a table from a tabulated
# colour to the flux of its smoothed profile within +-9 pixels (the trapezoid rule on X_FINE)
LSF = {}
INSIDE = {}
for nu in NU:
    LSF[nu] = RowellLSF(nu)
    INSIDE[nu] = float(np.trapezoid(TAB[nu], X_FINE))
# step 2, the table printed
for nu in NU:
    left_amplitude, right_amplitude = LSF[nu].edge         # the amplitudes A of the two wings [per pixel]
    profile_at_minus9 = RAW[nu][1][0]                      # L at the first digitized column
    profile_at_plus9 = RAW[nu][1][-1]                      # L at the last digitized column
    # the two wings beyond +-9 pixels hold the flux (left A + right A) x 9 pixels: the integral of A (9/|x|)^2
    # from 9 pixels to infinity is 9 A
    flux_in_tails = sum(LSF[nu].edge) * X_MAX
    print(f"nu_eff {nu:.2f}: L(-9), L(+9) = {profile_at_minus9:.4f}, {profile_at_plus9:.4f} "
          f"({profile_at_minus9 / ROW_LSF:.1f}, {profile_at_plus9 / ROW_LSF:.1f} image rows); "
          f"tail amplitudes {left_amplitude:.4f}, {right_amplitude:.4f}; "
          f"flux inside {INSIDE[nu]:.4f} + tails {flux_in_tails:.4f} = {LSF[nu].flux():.4f}")
# step 3, the checks: 94 to 95.5 % of the flux inside +-9 pixels; the total is one to within 1 %
assert all(0.94 <= INSIDE[nu] <= 0.955 for nu in NU)
assert all(abs(LSF[nu].flux() - 1.0) < 0.01 for nu in NU)

## 5. Interpolation in colour, and what the profiles say

Between two tabulated colours the profile is interpolated linearly in ν_eff:

$$L_\nu(x) = (1 - w)\,L_{\nu_a}(x) + w\,L_{\nu_b}(x),\qquad w = \frac{\nu - \nu_a}{\nu_b - \nu_a}.$$

**Its accuracy.** Leaving each interior colour out and predicting it from its
two neighbours, a step of 0.24 µm⁻¹, misses by about 1 % of the peak; over the
tabulated step of 0.12 µm⁻¹ the error is smaller. This is the footnote's
"accurate to about 1 % of the peak at best". Near the K star, the profiles at
1.36 and 1.48 µm⁻¹ differ by three to four times that, so the digitized data
resolve the colour dependence there.

**The width and the wings.** The FWHM falls from about 110 mas at
1.24 µm⁻¹ to about 100 mas at 1.48 µm⁻¹ and then stays (bluer light, narrower
profile). Within ±1 FWHM the profile holds 80 % of its flux, a Gaussian
98 %.

In [ ]:
# the interpolation's accuracy: each interior colour (k = 1, 2, 3 in NU) predicted from the mean of its two
# neighbours; and the difference of the two profiles next to the K star, 1.36 and 1.48
# step 1, LOO: a table from an interior tabulated colour to the largest error [per pixel] of its prediction from
# its two neighbours; LOO[1.36] is the error at 1.36, predicted from 1.24 and 1.48
LOO = {}
for k in (1, 2, 3):                        # k: the index in NU of an interior colour (1.36, 1.48, 1.60)
    predicted = 0.5 * (TAB[NU[k - 1]] + TAB[NU[k + 1]])    # the mean of the two neighbours
    LOO[NU[k]] = float(np.abs(predicted - TAB[NU[k]]).max())
LOO_MAX = max(LOO.values())
for nu, error in LOO.items():
    print(f"nu_eff {nu:.2f} predicted from its neighbours: largest error {error:.4f} per pixel "
          f"({error / TAB[nu].max():.1%} of the peak)")
# step 2, DIFF_K: the largest difference [per pixel] of the two tabulated profiles that bracket the K star
DIFF_K = float(np.abs(TAB[1.48] - TAB[1.36]).max())
print(f"1.36 against 1.48: largest difference {DIFF_K:.4f} per pixel, {DIFF_K / LOO_MAX:.2f} times the largest "
      f"interpolation error, {DIFF_K / LOO[1.36]:.2f} times that at 1.36, {DIFF_K / LOO[1.48]:.2f} at 1.48")
# step 3, the checks: every error below 0.005 per pixel and below 1.3 % of its profile's peak; the difference
# next to the K star three to four times the largest error
assert all(e < 0.005 for e in LOO.values()) and max(e / TAB[nu].max() for nu, e in LOO.items()) < 0.013
assert 3.0 <= DIFF_K / LOO_MAX <= 4.0

In [ ]:
# the width against colour, and the flux within one FWHM of the centre: the mask within_fwhm keeps the points of
# X_FINE there. NU[2:] omits the two reddest colours, so it keeps 1.48, 1.60 and 1.72
# step 1, FWHM: a table from a tabulated colour to its kernel's full width at half maximum [mas]; WITHIN: a table
# from a tabulated colour to the flux of its kernel within one FWHM of x = 0
FWHM = {}
WITHIN = {}
for nu in NU:
    FWHM[nu] = LSF[nu].fwhm_mas()
    within_fwhm = np.abs(X_FINE) <= FWHM[nu] / PIXEL_MAS   # the points of X_FINE within one FWHM of x = 0
    WITHIN[nu] = float(np.trapezoid(LSF[nu](X_FINE[within_fwhm]), X_FINE[within_fwhm]))
GAUSS_IN = 0.9815                          # erf(2 sqrt(ln 2)): a Gaussian's flux within +-1 FWHM
for nu in NU:
    print(f"nu_eff {nu:.2f}: FWHM {FWHM[nu]:6.1f} mas; flux within +-1 FWHM {WITHIN[nu]:.3f}")
# step 2, the checks (Sect. 6.2). round(x, -1) rounds to the nearest ten: 110 mas at 1.24, 100 mas from 1.48 on
assert round(FWHM[1.24], -1) == 110 and all(round(FWHM[nu], -1) == 100 for nu in NU[2:])
# the width falls by at least 5 mas from 1.24 to 1.48, and then changes by less than 2 mas
assert FWHM[1.24] - FWHM[1.48] >= 5.0 and max(FWHM[nu] for nu in NU[2:]) - min(FWHM[nu] for nu in NU[2:]) < 2.0
# 80 % of the flux within one FWHM (0.79 or 0.80 after rounding)
assert all(round(WITHIN[nu], 2) in (0.79, 0.80) for nu in NU)

In [ ]:
# the K star's kernel, L_K, at 1.42 per micron: between the tabulated 1.36 and 1.48
L_K = RowellLSF(NU_K)
print(f"the K star, nu_eff {NU_K}: FWHM {L_K.fwhm_mas():.1f} mas, flux {L_K.flux():.4f}")

## 6. The position derivative

A sample at offset x from the source receives the fraction L(x) of its flux.
The fits and the Fisher matrix need the derivative with respect to the
source position η: ∂L(x_j − η)/∂η = −L′(x)/PIXEL_MAS per mas. L′ is
analytic: the derivative of the core spline, of the tail, and of the blend
between them. It is checked against a central finite difference of the
kernel, inside the digitized range, across the blend and in the wings.

In [ ]:
# check the analytic position derivative L' against a central finite difference at nine offsets: in the core,
# across the blend and in the wings
check_offsets = np.array([-12.0, -8.0, -3.0, -0.5, 0.3, 2.5, 7.5, 8.9, 11.0])   # the nine offsets [pixels]
half_step = 1e-5                           # the half step of the finite difference [pixels]
# KERNELS: a table from a colour to its kernel: the five tabulated colours of LSF, and the K star's
KERNELS = dict(LSF)
KERNELS[NU_K] = L_K
# err: a table from a colour to the largest |L' - finite difference| over the nine offsets [per pixel^2]
err = {}
for nu, kernel in KERNELS.items():
    finite_difference = (kernel(check_offsets + half_step) - kernel(check_offsets - half_step)) / (2 * half_step)
    err[nu] = float(np.max(np.abs(finite_difference - kernel.derivative(check_offsets))))
for nu, largest_error in err.items():
    print(f"nu_eff {nu:.2f}: largest |L' - finite difference| {largest_error:.1e} per pixel^2")
assert max(err.values()) < 1e-8

In [ ]:
# the K star's L' on a fine grid: its largest slope, and its continuity across the blend (7 and 9 pixels)
offsets_fine = np.linspace(-12.0, 12.0, 24001)   # a grid of offsets [pixels], step 0.001 pixels
slope_K = L_K.derivative(offsets_fine)     # L' of the K star's kernel on that grid [per pixel^2]
largest_jump = np.abs(np.diff(slope_K)).max()    # the largest change of L' between neighbouring grid points
steepest = np.argmax(np.abs(slope_K))            # the index of the grid point with the largest |L'|
print(f"K star: largest slope {np.abs(slope_K).max():.4f} per pixel^2 at {offsets_fine[steepest]:+.2f} pixels; "
      f"continuous at 7 and 9 pixels: jumps {largest_jump:.1e} per grid step")
assert np.abs(np.diff(slope_K)).max() < 1e-3

## 7. The colour derivative

Because the profile is linear in ν between tabulated colours, its colour
derivative there is exact and constant: the secant of the two tabulated
profiles that bracket ν. For the K star, ν = 1.42 µm⁻¹, bracketed by 1.36 and
1.48:

$$\frac{\partial L}{\partial \nu}(x) = \frac{L_{1.48}(x) - L_{1.36}(x)}{1.48 - 1.36}\qquad (1.36 \le \nu \le 1.48).$$

Fig. 5 uses it for the colour column of the Fisher matrix and for panel (b):
a colour error Δν changes the profile by exactly Δν ∂L/∂ν. It is a property
of the linear interpolation, not of Gaia's LSF: Rowell's basis weights are
splines in ν, and the true derivative at 1.42 can differ. It changes at each
tabulated colour.

In [ ]:
# the colour derivative at the K star: the secant of the two tabulated profiles that bracket 1.42 per micron;
# index_below is the index in NU of the tabulated colour below the K star's
index_below = int(np.searchsorted(NU, NU_K)) - 1
NU_A = NU[index_below]                     # [1/micron] the tabulated colour below the K star's
NU_B = NU[index_below + 1]                 # [1/micron] the tabulated colour above the K star's
L_A = LSF[NU_A]                            # the kernel at NU_A
L_B = LSF[NU_B]                            # the kernel at NU_B


def dlsf_dnu(x_pix):
    """The colour derivative of the kernel between the two tabulated colours that bracket the K star: the secant
    of the two bracketing profiles. It is the same for every nu in [NU_A, NU_B].

    Inputs
        x_pix   the offset from the source [pixels]
    Returns dL/dnu [per pixel per (1/micron)]; positive: a bluer star (larger nu) has more light at that offset."""
    return (L_B(x_pix) - L_A(x_pix)) / (NU_B - NU_A)

In [ ]:
# the secant checked: the K star's kernel is the mix of the two profiles; a numerical derivative in nu; and the
# change that a colour error of +-0.01 makes
offsets_wide = np.linspace(-12.0, 12.0, 4801)   # a grid of offsets [pixels], step 0.005 pixels
colour_slope = dlsf_dnu(offsets_wide)      # dL/dnu on that grid [per pixel per (1/micron)]
COLOUR_ERROR = 0.01                        # [1/micron] the colour error of check 3 and of CHANGE_001
# check 1, the mix: the K star's kernel against (1 - weight_above) L_A + weight_above L_B, with weight_above the
# weight of the colour above; mix is their largest difference as a fraction of the kernel's peak
weight_above = (NU_K - NU_A) / (NU_B - NU_A)
mixed_profile = (1 - weight_above) * L_A(offsets_wide) + weight_above * L_B(offsets_wide)
mix = np.abs(L_K(offsets_wide) - mixed_profile).max() / L_K(offsets_wide).max()
# check 2, a numerical derivative in nu: the kernels at NU_K + 0.005 and NU_K - 0.005, their difference over 0.01
kernel_bluer = RowellLSF(NU_K + 0.005)
kernel_redder = RowellLSF(NU_K - 0.005)
central = (kernel_bluer(offsets_wide) - kernel_redder(offsets_wide)) / 0.01   # 0.01 = 2 x 0.005 [1/micron]
# check 3, step: a table from the sign of a colour error (-1 or +1) to the change of the profile that a colour
# error of that sign and of size 0.01 makes. round(..., 6) removes the rounding error of the sum
step = {}
for sign in (-1, 1):
    kernel_off = RowellLSF(round(NU_K + sign * COLOUR_ERROR, 6))
    step[sign] = kernel_off(offsets_wide) - L_K(offsets_wide)
# CHANGE_001: the largest change of the profile for a colour error of 0.01, as a fraction of the kernel's peak
CHANGE_001 = COLOUR_ERROR * np.abs(colour_slope).max() / L_K(offsets_wide).max()
print(f"bracketing colours {NU_A}, {NU_B}; the K star's kernel is their mix to {mix:.1e} of its peak")
largest_slope = np.abs(colour_slope).max()
at_largest = np.argmax(np.abs(colour_slope))     # the index of the grid point with the largest |dL/dnu|
print(f"dL/dnu: largest {largest_slope:.4f} per pixel per (1/micron), at {offsets_wide[at_largest]:+.2f} pixels; "
      f"a colour error of 0.01 changes the profile by at most {100 * CHANGE_001:.2f} % of the peak")
assert (NU_A, NU_B) == (1.36, 1.48) and mix < 1e-12
assert np.abs(central - colour_slope).max() < 1e-10 * np.abs(colour_slope).max()       # the numerical derivative
assert all(np.abs(step[sign] - sign * COLOUR_ERROR * colour_slope).max() < 1e-10 * np.abs(colour_slope).max()
           for sign in (-1, 1))                  # a colour error changes the profile by error x dL/dnu
assert CHANGE_001 < 0.003

## 8. The kernel on the toy window

The kernel replaces the Gaussian in the toy window: 18 samples of 59 mas,
the toy dictionary grid, the LSF's own FWHM as the resolution. The position
derivative enters as `dlsf_deta`. A source at the window centre leaves
94–95 % of its flux in the window (the Gaussian: all of it); the rest falls in
the wings outside, as on the sky.

In [ ]:
# the kernel on the toy window: instrument(nu) builds an ins.Instrument with the toy window's samples and trial
# positions, and with the Rowell LSF and its position derivative [per mas] as the kernel
def instrument(nu_eff):
    """The toy window with the Rowell LSF at nu_eff as its kernel."""
    L = RowellLSF(nu_eff)
    return ins.Instrument(sample_grid_mas=inst.sample_grid_mas, atom_eta_mas=inst.atom_eta_mas,
                      lsf=lambda d: L(np.asarray(d, dtype=float) / PIXEL_MAS),
                      dlsf_deta=lambda d: -L.derivative(np.asarray(d, dtype=float) / PIXEL_MAS) / PIXEL_MAS,
                      resolution_mas=L.fwhm_mas())

In [ ]:
# the K star's window, GAIA; and the fraction of a centred source's flux that falls in the 18 samples, per colour
GAIA = instrument(NU_K)
# CAPTURE: a table from a colour (the five tabulated ones, then the K star's) to that fraction.
# design_columns: the window's response to unit flux at each position (here one source at 0 mas), one number per
# sample; their sum is the part of the flux that the 18 samples hold
centred_source = np.array([0.0])           # one position: the window's centre [mas]
CAPTURE = {}
for nu in (*NU, NU_K):                     # (*NU, NU_K): the five tabulated colours and the K star's, one tuple
    response = instrument(nu).design_columns(centred_source)
    CAPTURE[nu] = float(response.sum())
for nu, captured in CAPTURE.items():
    print(f"nu_eff {nu:.2f}: a centred source leaves {captured:.3f} of its flux in the 18 samples")
print(f"the K star's window: resolution {GAIA.resolution_mas:.1f} mas")
assert all(0.93 <= c <= 0.96 for c in CAPTURE.values())

## 9. The figure

(a) The raw digitized FOV1 profiles (dots, every fourth column) and the
smoothed ones (lines). (b) The kernel on a logarithmic scale against the
offset in mas, the wings beyond ±9 pixels thin, the digitized range shaded;
the toy Gaussian dashed. (c) The position derivative L′ of the K star's
kernel. (d) The colour derivative ∂L/∂ν at the K star, in percent of its peak
per 0.01 µm⁻¹.

In [ ]:
# the figure, in the paper's style: (a) raw and smoothed profiles, (b) the kernel and its wings against the toy
# Gaussian, (c) L', (d) dL/dnu; written to pdf/ only with SAVE_PDF
PAPER_STYLE = {"font.size": 10.0, "xtick.labelsize": 9.0, "ytick.labelsize": 9.0, "axes.linewidth": 0.5,
               "xtick.direction": "in", "ytick.direction": "in"}
with plt.rc_context(PAPER_STYLE):                                         # the paper's style, this plot only
    # gauss_sigma: the sigma of the toy Gaussian LSF of FWHM 100 mas [pixels]
    gauss_sigma = 100.0 / (2.0 * np.sqrt(2.0 * np.log(2.0))) / PIXEL_MAS

    def gauss(x):
        """The toy Gaussian LSF of FWHM 100 mas [per pixel] at x [pixels], for comparison in panel (b)."""
        return np.exp(-0.5 * (x / gauss_sigma) ** 2) / (gauss_sigma * np.sqrt(2.0 * np.pi))

    offsets_panel_b = np.linspace(-16.0, 16.0, 2001)                      # the offsets of panel (b) [pixels]
    fig, ax = plt.subplots(1, 4, figsize=(14.0, 3.4))                     # ax[0] to ax[3]: panels (a) to (d)
    for nu in NU:
        offsets, profile = RAW[nu]                                        # the digitized offsets and LSF values
        # [::4] keeps every fourth column of the raw curve
        ax[0].plot(offsets[::4], profile[::4], ".", ms=1.5, color=COLOURS[nu])
        ax[0].plot(X_FINE, TAB[nu], color=COLOURS[nu], lw=0.8, label=f"{nu:.2f}")
        # panel (b): the kernel with its wings (thin), and within the digitized range (thick)
        ax[1].semilogy(offsets_panel_b * PIXEL_MAS, LSF[nu](offsets_panel_b), color=COLOURS[nu], lw=0.5)
        ax[1].semilogy(X_FINE * PIXEL_MAS, LSF[nu](X_FINE), color=COLOURS[nu], lw=1.0)
    # the toy Gaussian, dashed; np.clip(..., 1e-6, None) raises the values below 1e-6 to 1e-6 for the log scale
    ax[1].semilogy(offsets_panel_b * PIXEL_MAS, np.clip(gauss(offsets_panel_b), 1e-6, None), "k--", lw=0.8)
    ax[1].axvspan(-X_MAX * PIXEL_MAS, X_MAX * PIXEL_MAS, color="0.93", zorder=0)   # the digitized range, shaded
    # panel (c): L' of the K star on offsets_fine (section 6); panel (d): colour_slope on offsets_wide (section 7);
    # 100 x 0.01 x dL/dnu over the peak is the change in percent of the peak for a colour error of 0.01
    ax[2].plot(offsets_fine, L_K.derivative(offsets_fine), "k-", lw=0.8)
    ax[3].plot(offsets_wide, 100 * 0.01 * colour_slope / L_K(offsets_wide).max(), "k-", lw=0.8)
    # ax[2:] omits panels (a) and (b): panels (c) and (d) get a zero line and the limits of the blend
    for panel in ax[2:]:
        panel.axhline(0, color="0.6", lw=0.5)
        for blend_limit in (-X_JOIN, X_JOIN, -X_MAX, X_MAX):
            panel.axvline(blend_limit, color="0.85", lw=0.5, zorder=0)
    ax[0].set_xlim(-6, 6)
    ax[0].set_xlabel("AL offset [pixels]")
    ax[0].set_ylabel("LSF [per pixel]")
    ax[0].legend(title=r"$\nu_{\rm eff}$ [µm$^{-1}$]", fontsize=7, title_fontsize=7, frameon=False)
    ax[1].set_ylim(1e-4, 1)
    ax[1].set_xlim(-900, 900)
    ax[1].set_xlabel("AL offset [mas]")
    ax[2].set_xlabel("AL offset [pixels]")
    ax[2].set_ylabel(r"$L'$ [per pixel$^2$]")
    ax[3].set_xlabel("AL offset [pixels]")
    ax[3].set_ylabel(r"$0.01\,\partial L/\partial\nu$ [% of peak]")
    for panel, letter in zip(ax, "abcd"):
        panel.tick_params(direction="in", which="both")
        panel.text(0.04, 0.96, f"({letter})", transform=panel.transAxes, va="top", ha="left", fontsize=9,
                   fontweight="bold", bbox=dict(fc="white", ec="none", pad=1.0))
    fig.tight_layout()
    plt.show()

if SAVE_PDF:
    Path("pdf").mkdir(exist_ok=True)
    fig.savefig(Path("pdf") / "kernel_rowell_lsf.pdf", bbox_inches="tight")
    print("wrote pdf/kernel_rowell_lsf.pdf")
else:
    print("SAVE_PDF is False: the PDF is not written")